<a href="https://colab.research.google.com/github/FrontendDevSQ/Brain-Tumor-Classifier/blob/main/Brain_Tumor_Classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
print(torch.cuda.is_available())  # True if GPU enabled
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")



True
Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

os.listdir('/content/drive/MyDrive')


['Colab Notebooks',
 'Dataset',
 'FLX-S8-368.gdoc',
 'ShafaqAbdulMoiz_UpdatedResume.pdf',
 'ShafaqAbdulMoiz_Resume2.pdf',
 'ShafaqAbdulMoiz_Resume1.pdf',
 'SRS_Template.gdoc',
 'FLX-S8-368 table.xlsx',
 'Portfolio.gsite',
 'Saved from Chrome',
 'Google AI Studio',
 'ShafaqAbdulMoiz_Resume (3).pdf',
 'Shafaq_Abdul_Moiz_SQA_Resume.pdf',
 'base.apk.adding',
 'Resident connect.apk']

In [5]:
os.listdir('/content/drive/MyDrive/Dataset')


['Testing', 'Training']

In [6]:
train_path = '/content/drive/MyDrive/Dataset/Training'


In [7]:
test_path = '/content/drive/MyDrive/Dataset/Testing'

In [8]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

train_path = '/content/drive/MyDrive/Dataset/Training'
test_path = '/content/drive/MyDrive/Dataset/Testing'

tf = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dl = DataLoader(
    datasets.ImageFolder(train_path, tf),
    batch_size=32, shuffle=True, num_workers=2, pin_memory=True
)

test_dl = DataLoader(
    datasets.ImageFolder(test_path, tf),
    batch_size=32, shuffle=False, num_workers=2, pin_memory=True
)

In [9]:
model = nn.Sequential(
    # --- Convolutional Block 1 ---
    nn.Conv2d(3, 32, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),

    # --- Convolutional Block 2 ---
    nn.Conv2d(32, 64, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),

    # --- Convolutional Block 3 ---
    nn.Conv2d(64, 128, 3, 1, 1), nn.ReLU(), nn.MaxPool2d(2),

    # --- Transition to Dense Layers ---
    nn.Flatten(),

    # --- Fully Connected Layer 1 ---
    nn.Linear(128 * 16 * 16, 256), nn.ReLU(), nn.Dropout(0.5), # Corrected input size: 128 * 16 * 16

    # --- Output Layer ---
    nn.Linear(256, 4)  # 4 classes
).to(device)

In [10]:
opt = optim.AdamW(model.parameters(), 1e-5)
loss_fn = nn.CrossEntropyLoss()

In [12]:
model.train()

for epoch in range(25):
    running_loss = 0
    for batch_idx, (x, y) in enumerate(train_dl):
        opt.zero_grad()

        loss = loss_fn(model(x.to(device)), y.to(device))
        loss.backward()

        running_loss += loss.item() # Use .item() to get scalar value for accumulation

        opt.step()


    print(f'Epoch {epoch+1}: Total Loss = {running_loss:.4f}')

Epoch 1: Total Loss = 78.9507
Epoch 2: Total Loss = 76.0344
Epoch 3: Total Loss = 75.3118
Epoch 4: Total Loss = 74.5328
Epoch 5: Total Loss = 72.3723
Epoch 6: Total Loss = 71.3447
Epoch 7: Total Loss = 69.9510
Epoch 8: Total Loss = 69.1724
Epoch 9: Total Loss = 66.3079
Epoch 10: Total Loss = 65.6478
Epoch 11: Total Loss = 64.5271
Epoch 12: Total Loss = 64.1523
Epoch 13: Total Loss = 62.7794
Epoch 14: Total Loss = 61.0275
Epoch 15: Total Loss = 60.2861
Epoch 16: Total Loss = 59.2207
Epoch 17: Total Loss = 58.4307
Epoch 18: Total Loss = 56.9102
Epoch 19: Total Loss = 55.6278
Epoch 20: Total Loss = 55.7868
Epoch 21: Total Loss = 55.3493
Epoch 22: Total Loss = 53.0852
Epoch 23: Total Loss = 53.0332
Epoch 24: Total Loss = 52.2064
Epoch 25: Total Loss = 51.6985


In [13]:
model.eval() # Sets the model to evaluation mode (disables layers like Dropout)
test_loss = 0.0
correct = 0.0

with torch.no_grad(): # Disables gradient computation for faster inference and reduced memory
    for x, y in test_dl: # Iterates through the test data loader
        x, y = x.to(device), y.to(device) # Moves data to the appropriate device (CPU/GPU)

        logits = model(x) # Gets model predictions (logits)

        # Accumulates test loss (loss_fn must be defined in a previous cell)
        test_loss += loss_fn(logits, y).item() * y.size(0)

        # Calculates predictions
        preds = logits.argmax(dim=1) # Gets the index of the highest logit value (the predicted class)

        # Accumulates the number of correct predictions
        correct += (preds == y).sum().item()

# Calculate Average Loss
test_loss = test_loss / len(test_dl.dataset)


# Calculate Accuracy
accuracy = 100 * correct / len(test_dl.dataset)

# Print the results
print('Test Loss:', test_loss, 'Test accuracy', accuracy, '%')

KeyboardInterrupt: 

In [ ]:
import random
import matplotlib.pyplot as plt
from torchvision.transforms.functional import to_pil_image

model.eval() # Set the model to evaluation mode

# Select a random index from the test dataset
idx = random.randrange(len(test_dl.dataset))

# Get the image (img) and its true label
img, label = test_dl.dataset[idx]

# Denormalize the image for correct visualization (reversing the 0.5 mean/std normalization)
unnorm = img * 0.5 + 0.5

# Convert the tensor to a PIL image and plot it
plt.imshow(to_pil_image(unnorm))
plt.axis('off')
plt.title('Sample from test set')
plt.show()

with torch.no_grad(): # Disable gradient calculation for faster inference
    # Add a batch dimension (unsqueeze(0)) and move the image to the correct device
    logits = model(img.unsqueeze(0).to(device))

    # Get the index of the highest logit (the predicted class)
    pred = logits.argmax(dim=1).item()

# Get class names and print results
class_names = test_dl.dataset.classes
print(f'Predicted class: {class_names[pred]}')
print(f'Ground-truth: {class_names[label]}')